## Derived Metrics and Engineered Features
Engineering additional features from existing data to capture meaningful patterns that may not be directly visible from raw EDA is an essential practice in almost all analytics and model pipelines. These engineered features help improve analysis, uncover hidden relationships, and support deeper insights.

### Importing Libraries and Loading the Data
We load the processed clean dataset and import the libraries required for feature engineering.

In [1]:
import pandas as pd  # Core library for data manipulation and preprocessing
import numpy as np  # Numerical computing library used for arrays and mathematical operations
import matplotlib.pyplot as plt  # Plotting library for generating static visualisations
import seaborn as sns  # Statistical visualisation library built on top of matplotlib

In [2]:
df = pd.read_csv('clean_food_delivery_dataset.csv')  # Load the cleaned food delivery dataset into a pandas DataFrame

### Type-Driven Metrics
These are metrics that are applicable to features due to the data type of the features. These metrics do not require rigorous expertise in analytics or domain knowledge to derive meaningfully.

#### Aggregate-Based Metrics
We can derive aggregates of features, such as mean or median for numerical variables, and mode for categorical variables.

In [3]:
df['price'].mean()  # Mean price of orders in the dataset

224.64234285714286

In [4]:
df['customer_city'].value_counts()  # Frequency of orders in different cities in the dataset

customer_city
Mumbai       82
New Delhi    64
Pune         48
Nagpur       46
Hyderabad    45
Bengaluru    33
Name: count, dtype: int64

In [5]:
df['customer_city'].mode()  # City with most orders

0    Mumbai
Name: customer_city, dtype: object

In [6]:
df.groupby('customer_city')['price'].mean()  # Mean price of orders in the dataset by city

customer_city
Bengaluru    242.014545
Hyderabad    216.331111
Mumbai       217.765122
Nagpur       207.881957
New Delhi    236.448125
Pune         222.282292
Name: price, dtype: float64

#### DateTime-Based Metrics
We can extract meaningful components such as day of week, month, and hour from the order timestamp to support time-based behavioural and operational analysis.

In [7]:
df['order_date'] = pd.to_datetime(df['order_date'])  # Ensure datetime data type

In [8]:
# Extract day name, month, and hour as new time-based features
df['order_day'] = df['order_date'].dt.day_name()
df['order_month'] = df['order_date'].dt.month
df['order_hour'] = df['order_date'].dt.hour
df[['order_date', 'order_day', 'order_month', 'order_hour']].sample(5)

,order_date,order_day,order_month,order_hour
46,2025-01-10,Friday,1,0
348,2025-01-10,Friday,1,0
338,2025-11-09,Sunday,11,0
39,2025-01-10,Friday,1,0
18,2025-01-10,Friday,1,0


### Data-Driven Metrics
These are metrics that emerge from analysing the data and identifying patterns and relationships within it. These metrics require a good amount of analytics expertise to derive and make sense of. Domain knowledge helps.

#### Bucketing
Converting raw delivery times into meaningful speed categories can enable clearer performance segmentation.

In [9]:
# Categorising delivery time into fast, medium, or slow groups for clearer performance analysis
def speed_bucket(x):
    if pd.isna(x):
        return 'Unknown'
    elif x <= 20:
        return 'Fast'
    elif x <= 40:
        return 'Medium'
    else:
        return 'Slow'

df['delivery_speed_category'] = df['delivery_time_minutes'].apply(speed_bucket)
df[['delivery_time_minutes', 'delivery_speed_category']].sample(5)

,delivery_time_minutes,delivery_speed_category
322,15.0,Fast
293,5.0,Fast
63,41.0,Slow
344,81.0,Slow
171,5.0,Fast


#### Formula-Based Metrics
We can calculate the effective price a customer pays after applying the discount.

In [10]:
# Actual payable amount
df['final_price'] = df['price'] * (100 - df['discount']) / 100
df[['price', 'discount', 'final_price']].sample(5)

,price,discount,final_price
16,200.00,10.0,180.0000
231,200.00,10.0,180.0000
13,143.14,5.0,135.9830
290,249.56,15.0,212.1260
341,254.21,5.0,241.4995


### Business-Driven Metrics
Business-driven derived metrics are those that play an important role in the specific business domain, and require good expertise in both analytics and domain knowledge to engineer and apply suitably.

#### RFM Analysis
Recency (R), frequency (F), and monetary value (M) or RFM analysis centers around deriving RFM metrics that have been universally accepted as indicative of transactional businesses such as in the ecommerce domain. Recency means how recent was the customer last active. Frequency means how many times has the customer ordered from the business. Monetary value indicates the total revenue earned by the business from the customer.

In [11]:
# RFM analysis at a city level
rfm = df.groupby('customer_city').agg(recency = ('order_date', lambda x: (df['order_date'].max() - x.max()).days),
                                      frequency = ('order_id', 'count'),
                                      monetary = ('final_price', 'sum')).reset_index()

rfm

,customer_city,recency,frequency,monetary
0,Bengaluru,0,33,7314.5280
1,Hyderabad,60,45,9018.6725
2,Mumbai,90,82,16464.9555
3,Nagpur,61,46,8833.6280
4,New Delhi,62,64,14035.9310
5,Pune,31,48,9775.9965


### Export the Report
Save the processed data in CSV and Excel formats for downstream use.

In [12]:
rfm.to_csv('food_delivery_dataset_city_rfm.csv', index = False)  # CSV